# Crack-Distill Setup and Data Preparation 🛣️

This section initializes the repository structure, writes all code files, installs dependencies (including SAM 2), links inputs, and prepares the datasets.

In [1]:
# Create directory structure
!mkdir -p configs utils distillation scripts checkpoints data/datasets data/teacher_logits_box data/teacher_logits_centroid

In [2]:
%%writefile configs/config.yaml
# ============================================================
# Crack-Distill — Master Config
# ============================================================
# YOU ONLY NEED TO CHANGE:
#   1. data.datasets[0].path  → your crack500_yolo folder
#   2. student.backbone       → yolo11n-seg (demo) or yolo11s-seg (paper)
#   3. train.epochs           → 10 (demo) or 100 (real)
# Everything else leave as is.
# ============================================================

project:
  name: crack-distill
  seed: 42
  output_dir: runs/

# ----------------------------------------------------------
# TASK
# To switch: change type ONLY — everything else adapts.
# Options: instance_seg | semantic_seg | detection
# ----------------------------------------------------------
task:
  type: instance_seg
  num_classes: 1
  class_names:
    - crack

# ----------------------------------------------------------
# DATASET  ← change path here
# ----------------------------------------------------------
data:
  root: data/datasets/
  train_split: 0.8
  val_split: 0.1
  test_split: 0.1
  image_size: 512
  batch_size: 16
  num_workers: 4

  datasets:
    - name: combined
      path: data/datasets/combined_yolo
      format: yolo

# ----------------------------------------------------------
# TEACHER (SAM 2)  ← no changes needed
# ----------------------------------------------------------
teacher:
  model: sam2
  checkpoint: checkpoints/sam2_hiera_large.pt
  config: configs/sam2.1/sam2.1_hiera_l.yaml  # sam2 package config
  device: cuda
  prompt_type: box
  save_logits: true
  logits_dir: data/teacher_logits/
  batch_size: 4

# ----------------------------------------------------------
# STUDENT  ← change backbone here
# ----------------------------------------------------------
student:
  backbone: yolov8n-seg   # demo/iteration = yolo11n-seg — change to yolov8n-seg for paper
  pretrained: true
  device: "0,1"
  imgsz: 512

# ----------------------------------------------------------
# KD LOSSES  ← no changes needed
# L = L_task + α·L_mask_kd + β·L_feature + γ·L_boundary
# ----------------------------------------------------------
distillation:
  enabled: true
  temperature: 1.65      # lower = sharper soft targets, less NaN risk

  progressive:
    enabled: true
    stage1_pct: 0.30
    stage2_pct: 0.70

  losses:
    task:
      weight: 1.0
    mask_kd:
      enabled: true
      weight: 0.25       # now active (NaN bug fixed) — moderate KL signal
    feature:
      enabled: true
      weight: 0.18       # small — feature alignment should not dominate
      layers: [2, 5, 8]
    boundary:
      enabled: true
      weight: 2.0        # dominant — this is the only signal that worked before

# ----------------------------------------------------------
# TRAINING  ← change epochs here
# ----------------------------------------------------------
train:
  epochs: 150    # 10 for demo, 150 for real training
  lr: 0.001
  lr_scheduler: cosine
  warmup_epochs: 3
  optimizer: AdamW
  weight_decay: 0.0005
  amp: true
  grad_clip: 10.0
  save_every: 10
  patience: 20

# ----------------------------------------------------------
# EVALUATION  ← no changes needed
# ----------------------------------------------------------
eval:
  primary_metric: mAP50-seg
  metrics:
    - mAP50-seg
    - mAP50-95-seg
    - dice
    - boundary_iou
    - fps
    - latency_ms
    - params_M
    - gflops

# ----------------------------------------------------------
# PAPER EXPERIMENTS  ← no changes needed
# ----------------------------------------------------------
experiments:
  - name: baseline_finetune
    distillation.enabled: false
  - name: pseudo_labels
    distillation.losses.mask_kd.enabled: false
    distillation.losses.feature.enabled: false
    distillation.losses.boundary.enabled: false
  - name: full_kd_box
    distillation.enabled: true
    teacher.logits_dir: data/teacher_logits_box/
  - name: full_kd_centroid
    distillation.enabled: true
    teacher.logits_dir: data/teacher_logits_centroid/
  - name: low_data_5pct
    data.train_fraction: 0.05
  - name: low_data_10pct
    data.train_fraction: 0.10
  - name: low_data_25pct
    data.train_fraction: 0.25
  - name: low_data_50pct
    data.train_fraction: 0.50


Writing configs/config.yaml


In [3]:
%%writefile utils/config_loader.py
"""Config loader — converts YAML to a dot-access object."""

import yaml
from pathlib import Path


class ConfigNode:
    """Dot-access config object. cfg.data.batch_size just works."""

    def __init__(self, d: dict):
        for k, v in d.items():
            if isinstance(v, dict):
                setattr(self, k, ConfigNode(v))
            elif isinstance(v, list):
                setattr(self, k, [
                    ConfigNode(i) if isinstance(i, dict) else i for i in v
                ])
            else:
                setattr(self, k, v)

    def get(self, key, default=None):
        return getattr(self, key, default)

    def __contains__(self, key):
        return hasattr(self, key)

    def __repr__(self):
        return f"ConfigNode({self.__dict__})"

    def __iter__(self):
        return iter(self.__dict__.items())

    def dict(self):
        result = {}
        for k, v in self.__dict__.items():
            if isinstance(v, ConfigNode):
                result[k] = v.dict()
            elif isinstance(v, list):
                result[k] = [i.dict() if isinstance(i, ConfigNode) else i for i in v]
            else:
                result[k] = v
        return result


def load_config(path: str) -> ConfigNode:
    """Load YAML config and return dot-access ConfigNode."""
    with open(path) as f:
        raw = yaml.safe_load(f)
    return ConfigNode(raw)


def override_config(cfg: ConfigNode, overrides: dict) -> ConfigNode:
    """
    Apply flat-key overrides to a config.
    e.g. override_config(cfg, {"distillation.enabled": False})
    """
    raw = cfg.dict()
    for key_path, value in overrides.items():
        parts = key_path.split(".")
        node = raw
        for p in parts[:-1]:
            node = node.setdefault(p, {})
        node[parts[-1]] = value
    return ConfigNode(raw)


Writing utils/config_loader.py


In [4]:
%%writefile distillation/kd_trainer.py
"""
KD Trainer — correct implementation
=====================================
Processes soft logits and intermediate encoder features from SAM.
Registers hooks and trains 1x1 projection convolutions for feature distillation.
Uses picklable hooks and temporary hook/loss restoration during model saving to prevent pickling errors.
"""

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import OrderedDict
from ultralytics.models.yolo.segment.train import SegmentationTrainer


class KDYOLODataset(torch.utils.data.Dataset):
    """
    Wrapper for YOLO Dataset that preloads SAM teacher logits and features
    inside dataloader worker processes to hide disk I/O latency from GPU training.
    """
    def __init__(self, base_dataset, logits_dir, kd_cfg):
        self.base_dataset = base_dataset
        self.logits_dir = Path(logits_dir)
        self.features_dir = self.logits_dir.parent / "teacher_features"
        self.kd_cfg = kd_cfg

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        img_path = item.get("im_file", "")
        if not img_path:
            item["sam_target"] = None
            item["sam_feat"] = None
            return item
            
        stem = Path(img_path).stem

        sam_target = None
        sam_feat = None

        for prefix in [f"crack500_{stem}", f"deepcrack_{stem}", stem]:
            c = self.logits_dir / f"{prefix}_logits.npy"
            f_path = self.features_dir / f"{prefix}_features.npz"

            if c.exists():
                try:
                    raw = np.load(str(c))  # shape (M, 256, 256)
                    if raw.ndim == 2:
                        raw = np.expand_dims(raw, axis=0)
                    sam_target = torch.from_numpy(raw).float()
                except Exception:
                    pass

                if self.kd_cfg.losses.feature.enabled and f_path.exists():
                    try:
                        with np.load(str(f_path)) as data:
                            sam_feat = {
                                "image_embed": torch.from_numpy(data["image_embed"]).float(),
                                "feat1": torch.from_numpy(data["feat1"]).float()
                            }
                    except Exception:
                        pass
                break

        item["sam_target"] = sam_target
        item["sam_feat"] = sam_feat
        return item

    @property
    def collate_fn(self):
        return self.base_dataset.collate_fn

    def __getattr__(self, name):
        return getattr(self.base_dataset, name)


class ActiveHook:
    """
    A top-level picklable hook callback class.
    Writes features directly to a class attribute to avoid referencing local closures.
    """
    def __init__(self, key):
        self.key = key

    def __call__(self, module, input, output):
        KDSegmentationTrainer.student_features[self.key] = output


class KDSegmentationTrainer(SegmentationTrainer):
    # Class-level attribute to store hooked features safely
    student_features = {}

    def __init__(self, cfg, logits_dir=None, kd_cfg=None, **kwargs):
        super().__init__(cfg, **kwargs)
        
        # Under DDP, Ultralytics reinstantiates custom trainers using: CustomTrainer(cfg=cfg, overrides=overrides)
        # Since we avoid passing custom args in overrides to prevent get_cfg checks, we read them from environment variables.
        import os
        if logits_dir is None:
            logits_dir = os.environ.get("KD_LOGITS_DIR", "data/teacher_logits/")
                
        if kd_cfg is None:
            kd_config_json = os.environ.get("KD_CONFIG")
            if kd_config_json:
                try:
                    import json
                    from utils.config_loader import ConfigNode
                    kd_cfg = ConfigNode(json.loads(kd_config_json))
                except Exception:
                    pass
            
            if kd_cfg is None:
                # Under DDP, we can load configuration dynamically as a fallback
                from utils.config_loader import load_config
                try:
                    full_cfg = load_config("configs/config.yaml")
                    kd_cfg = full_cfg.distillation
                except Exception:
                    pass

        self.logits_dir  = Path(str(logits_dir))
        self.kd_cfg      = kd_cfg
        
        if kd_cfg is not None:
            self.temperature = float(kd_cfg.temperature)
            self.kd_weight   = float(kd_cfg.losses.boundary.weight)
        else:
            self.temperature = 1.6502
            self.kd_weight   = 2.0569
        
        self._current_paths = []
        self._kd_logged  = False
        self._no_logits_warned = False
        self.kd_losses   = []
        self._sam_targets = {}   # image_stem → soft target tensor (M, 256, 256)
        self._sam_features = {}  # image_stem → dict of features
        self._hook_handles = []

        print(f"[KD] logits_dir : {self.logits_dir}")
        print(f"[KD] logit files: {len(list(self.logits_dir.glob('*_logits.npy')))}")
        print(f"[KD] temperature: {self.temperature}")

    def setup_model(self):
        """Build model, set up projection layers and hooks, and call parent setup."""
        ckpt = super().setup_model()

        # Freezing segment head dynamically if progressive head freezing is enabled
        if hasattr(self.kd_cfg, "progressive") and self.kd_cfg.progressive.get("freeze_head", False):
            head_idx = 22
            try:
                from ultralytics.utils.torch_utils import unwrap_model
                model = unwrap_model(self.model)
            except Exception:
                model = self.model

            for idx, module in enumerate(model.model):
                if type(module).__name__ == "Segment":
                    head_idx = idx
                    break

            if self.args.freeze is None:
                self.args.freeze = [head_idx]
            elif isinstance(self.args.freeze, list):
                if head_idx not in self.args.freeze:
                    self.args.freeze.append(head_idx)
            elif isinstance(self.args.freeze, int):
                self.args.freeze = list(range(self.args.freeze))
                if head_idx not in self.args.freeze:
                    self.args.freeze.append(head_idx)
            print(f"[KD] Progressive: Freezing Segment head at index {head_idx}. args.freeze={self.args.freeze}")

        self._setup_proj_layers_and_hooks()
        self._patch_model_loss()
        return ckpt

    def _setup_proj_layers_and_hooks(self):
        """
        Dynamically determine student backbone feature shapes, initialize 1x1 convs
        for channel alignment, and register active training hooks.
        """
        try:
            from ultralytics.utils.torch_utils import unwrap_model
            model = unwrap_model(self.model)
        except Exception:
            model = self.model

        # Determine device
        device = next(self.model.parameters()).device

        # Standard layers to monitor
        layers_to_monitor = self.kd_cfg.losses.feature.layers if hasattr(self.kd_cfg.losses.feature, "layers") else [2, 5, 8]

        captured_shapes = {}
        def temp_hook(layer_idx):
            def hook(module, input, output):
                captured_shapes[layer_idx] = output.shape
            return hook

        hooks = []
        for idx in layers_to_monitor:
            if idx < len(model.model):
                h = model.model[idx].register_forward_hook(temp_hook(idx))
                hooks.append(h)

        # Run dummy forward pass to extract shapes
        dummy_input = torch.zeros((1, 3, self.args.imgsz, self.args.imgsz), device=device)
        model.eval()
        with torch.no_grad():
            try:
                _ = model(dummy_input)
            except Exception as e:
                print(f"[KD] Error during dummy forward pass for shapes: {e}")
        model.train()

        # Remove temporary hooks
        for h in hooks:
            h.remove()

        # Build projection layers
        proj_dict = nn.ModuleDict()
        for idx in layers_to_monitor:
            if idx in captured_shapes:
                in_channels = captured_shapes[idx][1]
                feature_h = captured_shapes[idx][2]
                stride = self.args.imgsz // feature_h
                out_channels = 64 if stride <= 4 else 256
                
                # Create a 1x1 Conv to align channels
                proj_dict[f"layer_{idx}"] = nn.Conv2d(in_channels, out_channels, kernel_size=1)
                print(f"[KD] Feature projection layer {idx}: stride {stride}, channels {in_channels} -> {out_channels}")

        # Register projection layers on the model so they are part of optimizer parameters
        model.add_module("proj_layers", proj_dict)
        self.proj_layers = proj_dict.to(device)

        # Register active training hooks
        self._register_active_hooks(model)

    def _register_active_hooks(self, model):
        """Helper to register forward hooks on target student model layers."""
        self._hook_handles.clear()
        KDSegmentationTrainer.student_features.clear()
        
        layers_to_monitor = self.kd_cfg.losses.feature.layers if hasattr(self.kd_cfg.losses.feature, "layers") else [2, 5, 8]
        for idx in layers_to_monitor:
            if idx < len(model.model):
                h = model.model[idx].register_forward_hook(ActiveHook(f"layer_{idx}"))
                self._hook_handles.append(h)
                print(f"[KD] Forward hook registered for layer {idx}")

    def save_model(self):
        """Override save_model to temporarily detach hooks and restore original loss function on both model and EMA model."""
        try:
            from ultralytics.utils.torch_utils import unwrap_model
            model = unwrap_model(self.model)
        except Exception:
            model = self.model

        # Get EMA model if defined
        ema_model = None
        if hasattr(self, "ema") and self.ema is not None and hasattr(self.ema, "ema"):
            try:
                ema_model = unwrap_model(self.ema.ema)
            except Exception:
                ema_model = self.ema.ema

        # Remove hooks on self.model
        for handle in self._hook_handles:
            handle.remove()
        self._hook_handles.clear()

        # Recursively clear forward hooks in all submodules for both models
        for m in [model, ema_model]:
            if m is not None:
                for submodule in m.modules():
                    submodule._forward_hooks.clear()

        # Restore original loss function if patched on both models
        original_loss_restored = False
        for m in [model, ema_model]:
            if m is not None and hasattr(m, "original_loss"):
                m.loss = m.original_loss
                original_loss_restored = True

        # Call original saving logic
        result = super().save_model()

        # Re-patch loss function on training model
        if original_loss_restored:
            self._patch_model_loss()

        # Re-register active hooks on training model
        self._register_active_hooks(model)
        
        # Reset the active hooks registered flag so that they get registered on the active training model again
        self._active_hooks_registered = False
        return result

    def build_dataset(self, img_path: str, mode: str = "train", batch: int | None = None):
        """Build custom KD dataset that wraps the default YOLO dataset."""
        base_dataset = super().build_dataset(img_path, mode, batch)
        if mode != "train":
            return base_dataset
        return KDYOLODataset(base_dataset, self.logits_dir, self.kd_cfg)

    def preprocess_batch(self, batch):
        """Preprocess batch and map preloaded SAM targets/features to GPU."""
        # Ensure active hooks are registered on the active running model (handles DDP deepcopy recreation)
        if not hasattr(self, "_active_hooks_registered") or not self._active_hooks_registered:
            try:
                from ultralytics.utils.torch_utils import unwrap_model
                active_model = unwrap_model(self.model)
            except Exception:
                active_model = self.model
            self._register_active_hooks(active_model)
            self._active_hooks_registered = True

        # Clear student features at the start of batch preprocessing
        KDSegmentationTrainer.student_features.clear()

        # Safety check to verify that all projection layer parameters are in the optimizer
        if not hasattr(self, "_checked_optimizer") and hasattr(self, "optimizer") and self.optimizer is not None:
            self._checked_optimizer = True
            proj_params = set(self.proj_layers.parameters())
            opt_params = set()
            for group in self.optimizer.param_groups:
                for p in group['params']:
                    opt_params.add(p)
            missing = proj_params - opt_params
            if missing:
                print(f"[KD] WARNING: {len(missing)} projection layer parameters are NOT in the optimizer! Training them will have no effect.")
            else:
                print("[KD] Success: All projection layer parameters are in the optimizer and will receive gradients.")

        batch = super().preprocess_batch(batch)
        im_files = batch.get("im_file", [])
        if isinstance(im_files, (str, Path)):
            im_files = [im_files]
        self._current_paths = list(im_files)

        # Retrieve the preloaded SAM targets and features from the batch dict
        sam_targets_list = batch.get("sam_target", [])
        sam_feats_list = batch.get("sam_feat", [])

        self._sam_targets = {}
        self._sam_features = {}

        for idx, img_path in enumerate(self._current_paths):
            stem = Path(img_path).stem
            
            if idx < len(sam_targets_list) and sam_targets_list[idx] is not None:
                # Clean NaNs and Infs to prevent nan mask_kd losses
                self._sam_targets[stem] = torch.nan_to_num(sam_targets_list[idx].to(self.device), nan=0.0, posinf=0.0, neginf=0.0)
                
            if idx < len(sam_feats_list) and sam_feats_list[idx] is not None:
                self._sam_features[stem] = {
                    k: torch.nan_to_num(v.to(self.device), nan=0.0, posinf=0.0, neginf=0.0) for k, v in sam_feats_list[idx].items()
                }

        if self._current_paths and not self._sam_targets and not self._no_logits_warned:
            stems = [Path(p).stem for p in self._current_paths[:3]]
            print(f"[KD] Warning: no SAM logits matched batch stems {stems}. "
                  f"Run: python scripts/generate_teacher_logits.py")
            self._no_logits_warned = True

        return batch

    def _patch_model_loss(self):
        """Patch model.loss() to add KD loss using student predictions."""
        trainer_ref = self

        try:
            from ultralytics.utils.torch_utils import unwrap_model
            model = unwrap_model(self.model)
        except Exception:
            model = self.model

        if not hasattr(model, "original_loss"):
            model.original_loss = model.loss

        original_loss_fn = model.original_loss.__func__ if hasattr(model.original_loss, "__func__") else None

        def patched_loss(self_model, batch, preds=None):
            if preds is None:
                preds = self_model.forward(batch["img"])

            if original_loss_fn is not None:
                base_loss, loss_items = original_loss_fn(self_model, batch, preds)
            else:
                base_loss, loss_items = type(self_model).loss(self_model, batch, preds)

            if not self_model.training:
                return base_loss, loss_items

            kd_losses = trainer_ref._kd_loss_from_preds(preds, batch, self_model)
            
            # Combine losses
            total = base_loss
            for k, v in kd_losses.items():
                total = total + v

            return total, loss_items

        import types
        model.loss = types.MethodType(patched_loss, model)
        print("[KD] model.loss() patched with detailed KD losses ✓")

    def _kd_loss_from_preds(self, preds, batch, model) -> dict:
        """
        Compute KL divergence, boundary, and feature alignment losses.
        """
        kd_losses = {}
        if not self._sam_targets:
            return kd_losses

        try:
            criterion = model.criterion
            preds_parsed = criterion.parse_output(preds)
            
            # Retrieve target assignments
            (fg_mask, target_gt_idx, target_bboxes, _, _), _, _ = criterion.get_assigned_targets_and_loss(preds_parsed, batch)
            
            pred_masks = preds_parsed["mask_coefficient"].permute(0, 2, 1).contiguous()
            proto = preds_parsed["proto"]
            
            loss_mask_kd = torch.tensor(0.0, device=self.device)
            loss_mask_kd_count = 0
            
            loss_boundary = torch.tensor(0.0, device=self.device)
            loss_boundary_count = 0

            # 1. Compute L_mask and L_boundary (Per-instance matched)
            for i, img_path in enumerate(self._current_paths):
                stem = Path(img_path).stem
                if stem not in self._sam_targets:
                    continue
                
                sam_logits = self._sam_targets[stem]
                fg_mask_i = fg_mask[i]
                
                if fg_mask_i.any() and sam_logits.shape[0] > 0:
                    mask_idx = target_gt_idx[i][fg_mask_i]
                    mask_idx = torch.clamp(mask_idx, 0, sam_logits.shape[0] - 1)
                    
                    # Compute student instance predicted mask logits: (N_pos, H_proto, W_proto)
                    pred_coefs = pred_masks[i][fg_mask_i]
                    pred_mask_logits = torch.einsum("in,nhw->ihw", pred_coefs, proto[i])
                    
                    # Extract corresponding SAM teacher logits: (N_pos, 256, 256)
                    sam_logits_matched = sam_logits[mask_idx]
                    
                    # Resize both to target resolution (256, 256)
                    student_mask_logits_resized = F.interpolate(
                        pred_mask_logits.unsqueeze(1),
                        size=(256, 256),
                        mode="bilinear",
                        align_corners=False
                    ).squeeze(1)
                    
                    sam_logits_matched_resized = F.interpolate(
                        sam_logits_matched.unsqueeze(1),
                        size=(256, 256),
                        mode="bilinear",
                        align_corners=False
                    ).squeeze(1)
                    
                    # Align dtypes to prevent precision/autocast mismatches
                    sam_logits_matched_resized = sam_logits_matched_resized.to(dtype=student_mask_logits_resized.dtype)
                    
                    # L_mask (KL Divergence on Bernoulli soft probabilities)
                    # FIX: clamp logits before sigmoid to prevent log(0) -> NaN
                    if self.kd_cfg.losses.mask_kd.enabled:
                        sam_clamped = torch.clamp(sam_logits_matched_resized / self.temperature, -15.0, 15.0)
                        stu_clamped = torch.clamp(student_mask_logits_resized / self.temperature, -15.0, 15.0)
                        q = torch.sigmoid(sam_clamped)
                        p_log = F.logsigmoid(stu_clamped)
                        inv_q = 1.0 - q
                        inv_p_log = F.logsigmoid(-stu_clamped)
                        
                        kl = q * (torch.log(q + 1e-8) - p_log) + inv_q * (torch.log(inv_q + 1e-8) - inv_p_log)
                        loss_mask_kd = loss_mask_kd + kl.mean() * (self.temperature ** 2)
                        loss_mask_kd_count += 1

                    # L_boundary (Per-instance matched boundary weighted loss)
                    if self.kd_cfg.losses.boundary.enabled:
                        sam_soft = torch.sigmoid(sam_logits_matched_resized / self.temperature)
                        bw = (1.0 - torch.abs(sam_soft - 0.5) * 2).detach()
                        bce = F.binary_cross_entropy_with_logits(
                            student_mask_logits_resized, sam_soft.detach(), reduction="none"
                        )
                        loss_boundary = loss_boundary + (bce * bw).mean()
                        loss_boundary_count += 1

            if self.kd_cfg.losses.mask_kd.enabled and loss_mask_kd_count > 0:
                kd_losses["mask_kd"] = (loss_mask_kd / loss_mask_kd_count) * self.kd_cfg.losses.mask_kd.weight
                
            if self.kd_cfg.losses.boundary.enabled and loss_boundary_count > 0:
                kd_losses["boundary"] = (loss_boundary / loss_boundary_count) * self.kd_cfg.losses.boundary.weight

            # 2. Compute L_feature (Scale-matched alignment)
            if self.kd_cfg.losses.feature.enabled:
                loss_feat = torch.tensor(0.0, device=self.device)
                layers_to_monitor = self.kd_cfg.losses.feature.layers if hasattr(self.kd_cfg.losses.feature, "layers") else [2, 5, 8]
                feat_count = 0
                
                for idx in layers_to_monitor:
                    feat_key = f"layer_{idx}"
                    if feat_key in self.student_features and feat_key in self.proj_layers:
                        sf = self.student_features[feat_key]
                        proj = self.proj_layers[feat_key]
                        sf_proj = proj(sf)
                        
                        # Map layer indices directly to SAM feature keys & channels (resolution/padding independent)
                        if idx == 2:
                            target_key = "feat1"
                            out_channels = 64
                        else:
                            target_key = "image_embed"
                            out_channels = 256
                        
                        feature_h = sf.shape[2]
                        
                        # Stack SAM features for the batch
                        tf_list = []
                        for img_path in self._current_paths:
                            stem = Path(img_path).stem
                            if stem in self._sam_features:
                                tf_list.append(self._sam_features[stem][target_key])
                            else:
                                tf_list.append(torch.zeros((1, out_channels, feature_h, feature_h), device=self.device))
                                
                        tf_batch = torch.cat(tf_list, dim=0).to(dtype=sf_proj.dtype)
                        
                        # Resize SAM feature spatially to match student feature
                        if sf_proj.shape[2:] != tf_batch.shape[2:]:
                            tf_batch = F.interpolate(tf_batch, size=sf_proj.shape[2:], mode="bilinear", align_corners=False)
                        
                        # FIX: normalize per-layer MSE so scale doesn't grow with channel dim
                        loss_feat = loss_feat + F.mse_loss(sf_proj, tf_batch.detach())
                        feat_count += 1
                    else:
                        if feat_key not in self.student_features and not self._no_logits_warned:
                            print(f"[KD] Warning: Hook feature {feat_key} not found in student_features. "
                                  f"Forward hooks might not be triggering. Skipping feature KD.")
                            self._no_logits_warned = True
                
                # FIX: average across layers so total feature loss is ~1 layer's worth, not 3×
                if feat_count > 0:
                    loss_feat = loss_feat / feat_count
                kd_losses["feature"] = loss_feat * self.kd_cfg.losses.feature.weight

            # Logging demonstration on first pass
            if not self._kd_logged and kd_losses:
                log_strs = [f"{k}: {float(v):.6f}" for k, v in kd_losses.items()]
                print(f"[KD] ✓ KD losses computed: {', '.join(log_strs)}")
                self._kd_logged = True

        except Exception as e:
            if not self._kd_logged:
                print(f"[KD] Warning: Error computing KD loss: {e} — skipping KD this batch")
                import traceback
                traceback.print_exc()
                self._kd_logged = True

        return kd_losses


Writing distillation/kd_trainer.py


In [5]:
%%writefile distillation/trainer.py
"""
KD Trainer — Crack-Distill
===========================
Uses KDSegmentationTrainer (subclass of YOLOv8 SegmentationTrainer)
to inject KD loss directly inside the training step.

L = L_task + γ · L_boundary_kd
"""

import json
import shutil
import numpy as np
import torch
from pathlib import Path

from utils.config_loader import load_config, override_config


def optuna_callback(trainer):
    """Callback to report validation metrics to Optuna and check for pruning."""
    trial = getattr(trainer, "optuna_trial", None)
    if trial is not None:
        metrics = getattr(trainer, "metrics", {})
        # YOLOv8-seg logs segment metrics to metrics/mAP50(M) or metrics/mAP50-95(M)
        score = metrics.get("metrics/mAP50(M)", 0.0)
        offset = getattr(trainer, "optuna_epoch_offset", 0)
        step = offset + trainer.epoch
        
        trial.report(score, step=step)
        
        # Check if Optuna recommends pruning this trial
        import optuna
        if trial.should_prune():
            print(f"[Optuna Callback] Pruning trial {trial.number} at step {step} with score {score:.4f}")
            raise optuna.exceptions.TrialPruned()


class CrackDistillTrainer:

    def __init__(self, cfg_path: str = "configs/config.yaml", override_cfg=None):
        self.cfg = override_cfg if override_cfg is not None else load_config(cfg_path)
        self.optuna_trial = None


        self.backbone    = self.cfg.student.backbone
        self.imgsz       = self.cfg.student.imgsz
        self.epochs      = self.cfg.train.epochs
        self.batch       = self.cfg.data.batch_size
        self.logits_dir  = Path(str(self.cfg.teacher.logits_dir)).expanduser().resolve()
        # On Kaggle, redirect to /tmp to avoid exceeding 20GB disk limit
        if "/kaggle/" in str(self.logits_dir):
            self.logits_dir = Path("/tmp") / self.logits_dir.name
        self.kd_enabled  = self.cfg.distillation.enabled
        self.temperature = self.cfg.distillation.temperature
        self.kd_weight   = self.cfg.distillation.losses.boundary.weight
        self.workers     = int(self.cfg.data.num_workers)
        self.device      = str(getattr(self.cfg.student, "device", "cuda"))

        # Dataset yaml
        ds  = self.cfg.data.datasets
        ds0 = ds[0] if isinstance(ds, list) else ds
        ds_path = Path(str(ds0.path if hasattr(ds0, "path") else ds0["path"])).expanduser().resolve()
        
        # If pointing to /kaggle/input, redirect to writable /kaggle/working symlink to use the patched dataset.yaml
        if "/kaggle/input/" in str(ds_path):
            ds_path = Path("/kaggle/working/data/datasets") / ds_path.name
            
        self.data_yaml = str(ds_path / "dataset.yaml")

        # Output dir — unique per experiment
        exp_name      = getattr(self.cfg.project, "experiment", "default")
        run_name      = f"{self.cfg.project.name}_{exp_name}_{self.cfg.task.type}_{self.backbone}"
        self.run_dir  = Path(self.cfg.project.output_dir).resolve() / run_name

        # Set environment variables for DDP processes to read trainer config safely
        import os
        os.environ["KD_LOGITS_DIR"] = str(self.logits_dir)
        os.environ["KD_CONFIG"] = json.dumps(self.cfg.distillation.dict())

        print(f"[Trainer] Task:       {self.cfg.task.type}")
        print(f"[Trainer] Student:    {self.backbone}")
        print(f"[Trainer] KD enabled: {self.kd_enabled}")
        print(f"[Trainer] Epochs:     {self.epochs}")
        print(f"[Trainer] Data:       {self.data_yaml}")
        print(f"[Trainer] Run dir:    {self.run_dir}")

    def train(self):
        from ultralytics import YOLO
        from ultralytics.cfg import get_cfg
        from distillation.kd_trainer import KDSegmentationTrainer

        progressive_cfg = getattr(self.cfg.distillation, "progressive", None)
        is_progressive = self.kd_enabled and progressive_cfg is not None and getattr(progressive_cfg, "enabled", False)

        if is_progressive:
            print(f"[Trainer] Progressive Distillation Enabled!")
            total_epochs = int(self.epochs)
            
            # Read percentage configuration (falling back to absolute values if needed)
            if hasattr(progressive_cfg, "stage1_pct"):
                stage1_pct = float(progressive_cfg.stage1_pct)
                stage1_epochs = max(1, int(round(total_epochs * stage1_pct)))
                stage2_epochs = max(1, total_epochs - stage1_epochs)
            else:
                stage1_epochs = int(getattr(progressive_cfg, "stage1_epochs", 3))
                stage2_epochs = int(getattr(progressive_cfg, "stage2_epochs", 7))
                
            print(f"[Trainer] Total Epochs: {total_epochs}")
            print(f"[Trainer] Stage 1 (Backbone): {stage1_epochs} epochs")
            print(f"[Trainer] Stage 2 (End-to-End): {stage2_epochs} epochs")

            # Clean old runs for this experiment
            for suffix in ["", "_stage1", "_stage2"]:
                old = self.run_dir.parent / f"{self.run_dir.name}{suffix}"
                if old.exists():
                    shutil.rmtree(old)
            self.run_dir.mkdir(parents=True, exist_ok=True)

            from utils.config_loader import ConfigNode

            # ================= STAGE 1 =================
            print("\n>>> STARTING PROGRESSIVE STAGE 1: Distill Backbone (Head Frozen) <<<")
            stage1_dict = self.cfg.distillation.dict()
            if "progressive" not in stage1_dict:
                stage1_dict["progressive"] = {}
            stage1_dict["progressive"]["freeze_head"] = True
            stage1_kd_cfg = ConfigNode(stage1_dict)

            overrides_stage1 = dict(
                data        = self.data_yaml,
                epochs      = stage1_epochs,
                imgsz       = self.imgsz,
                batch       = self.batch,
                device      = self.device,
                amp         = bool(self.cfg.train.amp),
                lr0         = float(self.cfg.train.lr),
                weight_decay= float(self.cfg.train.weight_decay),
                project     = str(self.run_dir.parent),
                name        = f"{self.run_dir.name}_stage1",
                exist_ok    = True,
                verbose     = True,
                model       = f"{self.backbone}.pt",
                workers     = self.workers,
            )
            cfg_obj_stage1 = get_cfg(overrides=overrides_stage1)
            trainer_stage1 = KDSegmentationTrainer(
                cfg         = cfg_obj_stage1,
                logits_dir  = self.logits_dir,
                kd_cfg      = stage1_kd_cfg,
            )
            if self.optuna_trial is not None:
                trainer_stage1.optuna_trial = self.optuna_trial
                trainer_stage1.optuna_epoch_offset = 0
                trainer_stage1.add_callback("on_fit_epoch_end", optuna_callback)
            trainer_stage1.train()

            # Find best weight of Stage 1
            stage1_run_dir = self.run_dir.parent / f"{self.run_dir.name}_stage1"
            stage1_best = stage1_run_dir / "weights/best.pt"
            if not stage1_best.exists():
                for candidate in sorted(self.run_dir.parent.glob(f"{self.run_dir.name}_stage1*/weights/best.pt")):
                    stage1_best = candidate
                    break
            print(f"[Trainer] Stage 1 finished. Best checkpoint loaded from: {stage1_best}")

            # ================= STAGE 2 =================
            print("\n>>> STARTING PROGRESSIVE STAGE 2: Distill Full Pipeline (Unfrozen) <<<")
            stage2_dict = self.cfg.distillation.dict()
            if "progressive" not in stage2_dict:
                stage2_dict["progressive"] = {}
            stage2_dict["progressive"]["freeze_head"] = False
            stage2_kd_cfg = ConfigNode(stage2_dict)

            overrides_stage2 = dict(
                data        = self.data_yaml,
                epochs      = stage2_epochs,
                imgsz       = self.imgsz,
                batch       = self.batch,
                device      = self.device,
                amp         = bool(self.cfg.train.amp),
                lr0         = float(self.cfg.train.lr),
                weight_decay= float(self.cfg.train.weight_decay),
                project     = str(self.run_dir.parent),
                name        = f"{self.run_dir.name}_stage2",
                exist_ok    = True,
                verbose     = True,
                model       = str(stage1_best),
                workers     = self.workers,
            )
            cfg_obj_stage2 = get_cfg(overrides=overrides_stage2)
            trainer_stage2 = KDSegmentationTrainer(
                cfg         = cfg_obj_stage2,
                logits_dir  = self.logits_dir,
                kd_cfg      = stage2_kd_cfg,
            )
            if self.optuna_trial is not None:
                trainer_stage2.optuna_trial = self.optuna_trial
                trainer_stage2.optuna_epoch_offset = stage1_epochs
                trainer_stage2.add_callback("on_fit_epoch_end", optuna_callback)
            trainer_stage2.train()

            # Find best weight of Stage 2
            stage2_run_dir = self.run_dir.parent / f"{self.run_dir.name}_stage2"
            stage2_best = stage2_run_dir / "weights/best.pt"
            if not stage2_best.exists():
                for candidate in sorted(self.run_dir.parent.glob(f"{self.run_dir.name}_stage2*/weights/best.pt")):
                    stage2_best = candidate
                    break

            # Copy stage 2 best weights to main run directory
            main_weights_dir = self.run_dir / "weights"
            main_weights_dir.mkdir(parents=True, exist_ok=True)
            self.best_pt = main_weights_dir / "best.pt"
            shutil.copy2(str(stage2_best), str(self.best_pt))
            print(f"[Trainer] Progressive training complete. Copied final weights to: {self.best_pt}")

        elif self.kd_enabled:
            # Clean old runs for this experiment
            for old in self.run_dir.parent.glob(f"{self.run_dir.name}*"):
                shutil.rmtree(old)
            self.run_dir.mkdir(parents=True, exist_ok=True)

            print(f"[Trainer] Using KDSegmentationTrainer")

            # Build overrides dict for YOLO
            overrides = dict(
                data        = self.data_yaml,
                epochs      = self.epochs,
                imgsz       = self.imgsz,
                batch       = self.batch,
                device      = self.device,
                amp         = bool(self.cfg.train.amp),
                lr0         = float(self.cfg.train.lr),
                weight_decay= float(self.cfg.train.weight_decay),
                project     = str(self.run_dir.parent),
                name        = self.run_dir.name,
                exist_ok    = True,
                verbose     = True,
                model       = f"{self.backbone}.pt",
                workers     = self.workers,
            )
 
            # Create KD trainer directly
            cfg_obj = get_cfg(overrides=overrides)
            trainer = KDSegmentationTrainer(
                cfg         = cfg_obj,
                logits_dir  = self.logits_dir,
                kd_cfg      = self.cfg.distillation,
            )
            if self.optuna_trial is not None:
                trainer.optuna_trial = self.optuna_trial
                trainer.optuna_epoch_offset = 0
                trainer.add_callback("on_fit_epoch_end", optuna_callback)
            trainer.train()

        else:
            print(f"[Trainer] Baseline — no KD")
            model = YOLO(f"{self.backbone}.pt")
            model.train(
                data        = self.data_yaml,
                epochs      = self.epochs,
                imgsz       = self.imgsz,
                batch       = self.batch,
                device      = self.device,
                amp         = bool(self.cfg.train.amp),
                lr0         = float(self.cfg.train.lr),
                weight_decay= float(self.cfg.train.weight_decay),
                project     = str(self.run_dir.parent),
                name        = self.run_dir.name,
                exist_ok    = True,
                verbose     = True,
                workers     = self.workers,
            )

        # Find best.pt
        self.best_pt = self.run_dir / "weights/best.pt"
        if not self.best_pt.exists():
            for candidate in sorted(self.run_dir.parent.glob(
                    f"{self.run_dir.name}*/weights/best.pt")):
                self.best_pt = candidate
                break

        print(f"[Trainer] Best model: {self.best_pt}")

    def test(self) -> dict:
        from ultralytics import YOLO

        if not hasattr(self, "best_pt") or not self.best_pt.exists():
            self.best_pt = self.run_dir / "weights/best.pt"
            if not self.best_pt.exists():
                for candidate in sorted(self.run_dir.parent.glob(
                        f"{self.run_dir.name}*/weights/best.pt")):
                    self.best_pt = candidate
                    break

        if not self.best_pt.exists():
            print("[Trainer] No trained model found.")
            return {}

        model   = YOLO(str(self.best_pt))
        metrics = model.val(
            data    = self.data_yaml,
            imgsz   = self.imgsz,
            device  = "cuda:0" if "cuda" in self.device or "," in self.device else self.device,
            verbose = True,
        )

        results = {
            "mAP50-box":    float(metrics.box.map50),
            "mAP50-seg":    float(metrics.seg.map50),
            "mAP50-95-seg": float(metrics.seg.map),
            "model":        str(self.best_pt),
        }

        out = self.run_dir / "test_results.json"
        with open(out, "w") as f:
            json.dump(results, f, indent=2)

        print("\n[Test Results]")
        for k, v in results.items():
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
        print(f"  Saved to: {out}")

        return results


Writing distillation/trainer.py


In [6]:
%%writefile scripts/convert_crack500.py
#!/usr/bin/env python3
"""
Crack500 → YOLO seg format converter
=====================================
Crack500 structure:
  crack500/
  ├── traincrop/   ← 00001.jpg + 00001.png (binary mask, same stem)
  ├── valcrop/
  ├── testcrop/
  ├── train.txt    ← list of image filenames (optional)
  ├── val.txt
  └── test.txt

Output (YOLO seg format, ready for ultralytics):
  crack500_yolo/
  ├── images/
  │   ├── train/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── train/
  │   ├── val/
  │   └── test/
  └── dataset.yaml

Each .txt label: one line per connected crack instance
  0 x1 y1 x2 y2 ... (normalized polygon, class 0 = crack)

Usage:
  python scripts/convert_crack500.py \
      --src ~/distill/data/datasets/crack500 \
      --dst ~/distill/data/datasets/crack500_yolo
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int) -> list[str]:
    """
    Read binary PNG mask → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.

    Returns list of label lines (one per instance).
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    # Threshold (Crack500 masks are 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path, split_name: str):
    """Process one split (train/val/test)."""

    # Crack500 stores images+masks together in traincrop/valcrop/testcrop
    crop_dir = src_dir / f"{split_name}crop"
    if not crop_dir.exists():
        # Try alternate names
        for candidate in [src_dir / split_name, src_dir / f"{split_name}data"]:
            if candidate.exists():
                crop_dir = candidate
                break
        else:
            print(f"  [WARNING] Could not find directory for split '{split_name}', skipping.")
            return 0

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all images (jpg/jpeg/png that are NOT masks)
    all_files = sorted(crop_dir.iterdir())
    # Crack500: image = .jpg, mask = same stem + .png
    image_files = [f for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')
                   and ':Zone.Identifier' not in f.name]

    if not image_files:
        # Some versions store as .png images too — distinguish by paired files
        png_files = [f for f in all_files if f.suffix.lower() == '.png'
                     and ':Zone.Identifier' not in f.name]
        # If .jpg exists for a stem → .png is mask. If no .jpg → .png is image.
        jpg_stems = {f.stem for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')}
        image_files = [f for f in png_files if f.stem not in jpg_stems]

    converted = 0
    skipped   = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find corresponding mask (.png with same stem)
        mask_path = crop_dir / f"{stem}.png"
        if not mask_path.exists():
            # Try .bmp
            mask_path = crop_dir / f"{stem}.bmp"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Convert mask to YOLO seg labels
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def write_dataset_yaml(dst: Path, num_train: int, num_val: int, num_test: int):
    """Write ultralytics-compatible dataset.yaml."""
    yaml_content = f"""# Crack500 — YOLO seg format
# Auto-generated by convert_crack500.py

path: {dst.resolve()}
train: images/train
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# train: ~{num_train} images
# val:   ~{num_val} images
# test:  ~{num_test} images
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")


def verify_conversion(dst: Path):
    """Quick sanity check on converted dataset."""
    print("\n[Verify] Checking converted dataset...")
    issues = 0
    for split in ["train", "val", "test"]:
        img_dir = dst / "images" / split
        lbl_dir = dst / "labels" / split
        if not img_dir.exists():
            continue

        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        lbls = list(lbl_dir.glob("*.txt"))

        # Check counts match
        if len(imgs) != len(lbls):
            print(f"  [!] {split}: {len(imgs)} images vs {len(lbls)} labels — mismatch!")
            issues += 1
        else:
            print(f"  {split}: {len(imgs)} images ✓")

        # Check a few labels are non-empty
        non_empty = sum(1 for l in lbls if l.stat().st_size > 0)
        empty     = len(lbls) - non_empty
        print(f"    labels with cracks: {non_empty} | empty (no crack): {empty}")

        if non_empty == 0:
            print(f"  [!] {split}: ALL labels are empty — check mask paths!")
            issues += 1

    if issues == 0:
        print("\n  ✓ Dataset looks good!")
    else:
        print(f"\n  ✗ {issues} issue(s) found — check output above.")

    return issues == 0


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        required=True,
        help="Path to crack500 root dir (contains traincrop/, valcrop/, testcrop/)"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default=None,
        help="Output directory (default: <src>_yolo)"
    )
    parser.add_argument(
        "--verify",
        action="store_true",
        default=True,
        help="Run sanity check after conversion"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve() if args.dst else src.parent / f"{src.name}_yolo"

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    counts = {}
    for split in ["train", "val", "test"]:
        n = process_split(
            src_dir    = src,
            dst_img_dir= dst / "images" / split,
            dst_lbl_dir= dst / "labels" / split,
            split_name = split,
        )
        counts[split] = n

    write_dataset_yaml(dst, counts["train"], counts["val"], counts["test"])

    if args.verify:
        verify_conversion(dst)

    print(f"\n[Done] Converted dataset at: {dst}")
    print(f"\nNext step — test YOLO11 loads it:")
    print(f"  from ultralytics import YOLO")
    print(f"  model = YOLO('yolo11n-seg.pt')")
    print(f"  model.train(data='{dst}/dataset.yaml', epochs=1, imgsz=512)")


if __name__ == "__main__":
    main()
# (appended — nothing, file is complete)


Writing scripts/convert_crack500.py


In [7]:
%%writefile scripts/convert_crack500_uncropped.py
#!/usr/bin/env python3
"""
Crack500 Uncropped Test/Val → YOLO seg format converter
======================================================
Converts the original uncropped validation and test sets of Crack500.
Handles EXIF orientation for images by rotating the corresponding masks.

Source directories:
  data/datasets/crack500/valdata/   ← contains {stem}.jpg and {stem}_mask.png
  data/datasets/crack500/testdata/  ← contains {stem}.jpg and {stem}_mask.png

Output (YOLO seg format):
  data/datasets/crack500_uncropped_yolo/
  ├── images/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── val/
  │   └── test/
  └── dataset.yaml
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm
from PIL import Image


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def get_exif_rotation(img_path: Path):
    """Retrieve EXIF orientation tag from image."""
    try:
        with Image.open(img_path) as im:
            exif = im.getexif()
            if exif:
                return exif.get(274)  # 274 is the Orientation tag
    except Exception:
        pass
    return None


def rotate_mask_to_match_image(mask: np.ndarray, exif_orientation: int) -> np.ndarray:
    """Rotate mask array to match image rotation applied by cv2.imread based on EXIF."""
    if exif_orientation == 6:
        return cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif exif_orientation == 8:
        return cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif exif_orientation == 3:
        return cv2.rotate(mask, cv2.ROTATE_180)
    return mask


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int, exif_orientation: int = None) -> list[str]:
    """
    Read binary PNG mask → rotate based on EXIF → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    if exif_orientation:
        mask = rotate_mask_to_match_image(mask, exif_orientation)

    # Threshold (Crack500 masks are binary 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_dir: Path, split_name: str):
    """Process uncropped val or test split."""
    split_dir = src_dir / f"{split_name}data"
    if not split_dir.exists():
        print(f"  [Warning] Directory {split_dir} does not exist, skipping split {split_name}.")
        return 0

    dst_img_dir = dst_dir / "images" / split_name
    dst_lbl_dir = dst_dir / "labels" / split_name

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all image files (jpg/jpeg/png that do not contain '_mask')
    all_files = sorted(split_dir.iterdir())
    image_files = [
        f for f in all_files 
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and '_mask' not in f.name.lower()
        and ':Zone.Identifier' not in f.name
    ]

    converted = 0
    skipped = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find mask (stem + "_mask.png")
        mask_path = split_dir / f"{stem}_mask.png"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions (matches how cv2.imread auto-rotates it based on EXIF)
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Get EXIF rotation from image
        exif_orientation = get_exif_rotation(img_path)

        # Convert mask to YOLO seg labels (rotating it to match)
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h, exif_orientation)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 Uncropped splits to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        default="data/datasets/crack500",
        help="Path to crack500 root dir"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default="data/datasets/crack500_uncropped_yolo",
        help="Output directory"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve()

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    if dst.exists():
        print(f"[Warning] Output directory exists, clearing: {dst}")
        shutil.rmtree(dst)

    dst.mkdir(parents=True, exist_ok=True)

    counts = {}
    for split in ["val", "test"]:
        n = process_split(src, dst, split)
        counts[split] = n

    # Write dataset_uncropped.yaml
    yaml_content = f"""# Crack500 Uncropped — YOLO seg format
# Auto-generated by convert_crack500_uncropped.py

path: {dst.resolve()}
train: images/val
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# val:   ~{counts.get('val', 0)} images (uncropped)
# test:  ~{counts.get('test', 0)} images (uncropped)
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")
    print(f"[Done] Converted uncropped splits successfully.")


if __name__ == "__main__":
    main()


Writing scripts/convert_crack500_uncropped.py


In [8]:
%%writefile scripts/convert_deepcrack.py
#!/usr/bin/env python3
"""
DeepCrack → YOLO seg format converter
=====================================
DeepCrack structure:
  deepcrack/
  ├── train_img/      ← 11111.jpg
  ├── train_lab/      ← 11111.png (binary mask, 0/255)
  ├── test_img/       ← 111212-1.jpg
  └── test_lab/       ← 111212-1.png (binary mask, 0/255)

Output (YOLO seg format, ready for ultralytics):
  deepcrack_yolo/
  ├── images/
  │   ├── train/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── train/
  │   ├── val/
  │   └── test/
  └── dataset.yaml

Each .txt label: one line per connected crack instance
  0 x1 y1 x2 y2 ... (normalized polygon, class 0 = crack)

Splits:
  - Train: 80% of train_img (480 images)
  - Val: 20% of train_img (120 images)
  - Test: 100% of test_img (474 images)
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int) -> list[str]:
    """
    Read binary PNG mask → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    # Threshold (DeepCrack masks are 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_images(image_list: list[Path], mask_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path, split_name: str):
    """Process a list of images for a specific split."""
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    converted = 0
    skipped = 0

    for img_path in tqdm(image_list, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find corresponding mask (.png with same stem in mask_dir)
        mask_path = mask_dir / f"{stem}.png"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Convert mask to YOLO seg labels
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def write_dataset_yaml(dst: Path, num_train: int, num_val: int, num_test: int):
    """Write ultralytics-compatible dataset.yaml."""
    yaml_content = f"""# DeepCrack — YOLO seg format
# Auto-generated by convert_deepcrack.py

path: {dst.resolve()}
train: images/train
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# train: ~{num_train} images
# val:   ~{num_val} images
# test:  ~{num_test} images
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")


def verify_conversion(dst: Path):
    """Quick sanity check on converted dataset."""
    print("\n[Verify] Checking converted dataset...")
    issues = 0
    for split in ["train", "val", "test"]:
        img_dir = dst / "images" / split
        lbl_dir = dst / "labels" / split
        if not img_dir.exists():
            continue

        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        lbls = list(lbl_dir.glob("*.txt"))

        # Check counts match
        if len(imgs) != len(lbls):
            print(f"  [!] {split}: {len(imgs)} images vs {len(lbls)} labels — mismatch!")
            issues += 1
        else:
            print(f"  {split}: {len(imgs)} images ✓")

        # Check a few labels are non-empty
        non_empty = sum(1 for l in lbls if l.stat().st_size > 0)
        empty     = len(lbls) - non_empty
        print(f"    labels with cracks: {non_empty} | empty (no crack): {empty}")

        if non_empty == 0:
            print(f"  [!] {split}: ALL labels are empty — check mask paths!")
            issues += 1

    if issues == 0:
        print("\n  ✓ Dataset looks good!")
    else:
        print(f"\n  ✗ {issues} issue(s) found — check output above.")

    return issues == 0


def main():
    parser = argparse.ArgumentParser(description="Convert DeepCrack to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        default="data/datasets/deepcrack",
        help="Path to deepcrack root dir"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default=None,
        help="Output directory (default: <src>_yolo)"
    )
    parser.add_argument(
        "--verify",
        action="store_true",
        default=True,
        help="Run sanity check after conversion"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve() if args.dst else src.parent / f"{src.name}_yolo"

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    # Check and clean output directory if exists
    if dst.exists():
        print(f"[Warning] Output directory exists, clearing: {dst}")
        shutil.rmtree(dst)

    # 1. Process Train split and do 80-20 partition
    train_img_dir = src / "train_img"
    train_lab_dir = src / "train_lab"
    
    if not train_img_dir.exists() or not train_lab_dir.exists():
        print(f"ERROR: Train directories not found under {src}")
        return

    all_train_files = sorted([
        f for f in train_img_dir.iterdir()
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and ':Zone.Identifier' not in f.name
    ])

    # Deterministic shuffle
    rng = np.random.RandomState(42)
    shuffled_indices = rng.permutation(len(all_train_files))
    
    n_train = int(len(all_train_files) * 0.8)
    
    train_files = [all_train_files[i] for i in shuffled_indices[:n_train]]
    val_files   = [all_train_files[i] for i in shuffled_indices[n_train:]]

    print(f"Total training images: {len(all_train_files)}")
    print(f"  -> Train split: {len(train_files)}")
    print(f"  -> Val split:   {len(val_files)}")

    # 2. Process Test split
    test_img_dir = src / "test_img"
    test_lab_dir = src / "test_lab"
    
    if not test_img_dir.exists() or not test_lab_dir.exists():
        print(f"ERROR: Test directories not found under {src}")
        return

    test_files = sorted([
        f for f in test_img_dir.iterdir()
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and ':Zone.Identifier' not in f.name
    ])
    print(f"Total test images:     {len(test_files)}")

    # Convert splits
    counts = {}
    counts["train"] = process_images(
        image_list = train_files,
        mask_dir   = train_lab_dir,
        dst_img_dir= dst / "images" / "train",
        dst_lbl_dir= dst / "labels" / "train",
        split_name = "train",
    )
    
    counts["val"] = process_images(
        image_list = val_files,
        mask_dir   = train_lab_dir,
        dst_img_dir= dst / "images" / "val",
        dst_lbl_dir= dst / "labels" / "val",
        split_name = "val",
    )

    counts["test"] = process_images(
        image_list = test_files,
        mask_dir   = test_lab_dir,
        dst_img_dir= dst / "images" / "test",
        dst_lbl_dir= dst / "labels" / "test",
        split_name = "test",
    )

    write_dataset_yaml(dst, counts["train"], counts["val"], counts["test"])

    if args.verify:
        verify_conversion(dst)

    print(f"\n[Done] Converted DeepCrack dataset at: {dst}")


if __name__ == "__main__":
    main()


Writing scripts/convert_deepcrack.py


In [9]:
%%writefile scripts/generate_teacher_logits.py
#!/usr/bin/env python3
"""
Generate SAM 2 teacher logits for crack500 training images.
Run ONCE before training. Saves .npy logit files to data/teacher_logits/

Usage (from ~/distill):
  python scripts/generate_teacher_logits.py           # full dataset
  python scripts/generate_teacher_logits.py --resume  # skip already done
  python scripts/generate_teacher_logits.py --max 50  # only first N images
"""

import argparse
import sys
import os
import cv2
import numpy as np
import torch
from pathlib import Path

# ── project root on path ─────────────────────────────────────
ROOT = Path(__file__).parent.parent.resolve()
sys.path.insert(0, str(ROOT))

# ── fixed paths (relative to ~/distill) ─────────────────────
DATASET_DIR = ROOT / "data/datasets/crack500"
LOGITS_DIR  = ROOT / "data/teacher_logits"
SAM2_CKPT   = ROOT / "checkpoints/sam2_hiera_large.pt"
SAM2_CFG    = "configs/sam2.1/sam2.1_hiera_l.yaml"


def mask_to_bbox(mask: np.ndarray):
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any():
        return None
    r1, r2 = np.where(rows)[0][[0, -1]]
    c1, c2 = np.where(cols)[0][[0, -1]]
    return np.array([c1, r1, c2, r2], dtype=np.float32)


def mask_to_centroid(mask: np.ndarray):
    M = cv2.moments(mask)
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
        h, w = mask.shape[:2]
        if 0 <= cX < w and 0 <= cY < h and mask[cY, cX] > 0:
            return np.array([[cX, cY]], dtype=np.float32)
    # Fallback to maximum of distance transform (guaranteed to be inside mask)
    dist_transform = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    _, _, _, max_loc = cv2.minMaxLoc(dist_transform)
    cX, cY = max_loc
    return np.array([[cX, cY]], dtype=np.float32)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--resume", action="store_true",
                        help="Skip images that already have logits")
    parser.add_argument("--max", type=int, default=None,
                        help="Max images to process (for testing)")
    parser.add_argument("--img-dir", type=str, default=None,
                        help="Path to directory of training images")
    parser.add_argument("--mask-dir", type=str, default=None,
                        help="Path to directory of training masks")
    parser.add_argument("--prefix", type=str, default="crack500_",
                        help="Prefix to use for saved logits files")
    parser.add_argument("--prompt-type", type=str, default="box_centroid",
                        choices=["box", "box_centroid"],
                        help="Prompt type to use (box or box_centroid)")
    parser.add_argument("--logits-dir", type=str, default=None,
                        help="Override output logits directory")
    args = parser.parse_args()

    if args.logits_dir:
        logits_dir = Path(args.logits_dir).expanduser().resolve()
    else:
        logits_dir = ROOT / "data/teacher_logits"
        
    # On Kaggle, redirect to /tmp/ to avoid exceeding 20GB disk limit
    if "/kaggle/" in str(logits_dir):
        logits_dir = Path("/tmp") / logits_dir.name
        
    logits_dir.mkdir(parents=True, exist_ok=True)

    if args.img_dir:
        # Single custom dataset mode
        datasets = [{
            "name": "Custom",
            "img_dir": Path(args.img_dir).expanduser().resolve(),
            "mask_dir": Path(args.mask_dir).expanduser().resolve() if args.mask_dir else Path(args.img_dir).expanduser().resolve(),
            "prefix": args.prefix
        }]
    else:
        # Multi-dataset auto-mode
        datasets = [
            {
                "name": "Crack500",
                "img_dir": ROOT / "data/datasets/crack500/traincrop",
                "mask_dir": ROOT / "data/datasets/crack500/traincrop",
                "prefix": "crack500_"
            },
            {
                "name": "DeepCrack",
                "img_dir": ROOT / "data/datasets/deepcrack/train_img",
                "mask_dir": ROOT / "data/datasets/deepcrack/train_lab",
                "prefix": "deepcrack_"
            }
        ]

    # Filter out non-existent datasets
    active_datasets = []
    for d in datasets:
        if d["img_dir"].exists():
            active_datasets.append(d)
        else:
            if args.img_dir:
                print(f"ERROR: Image directory not found: {d['img_dir']}")
                return
            else:
                print(f"Skipping {d['name']} logits generation: directory {d['img_dir']} does not exist.")

    if not active_datasets:
        print("No active datasets to process. Exiting.")
        return

    # ── Load SAM 2 ───────────────────────────────────────────
    print(f"Loading SAM 2 from {SAM2_CKPT}...")
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor

    model     = build_sam2(SAM2_CFG, str(SAM2_CKPT), device="cuda")
    predictor = SAM2ImagePredictor(model)
    print("SAM 2 loaded ✓\n")

    for d in active_datasets:
        img_dir = d["img_dir"]
        mask_dir = d["mask_dir"]
        prefix = d["prefix"]

        # ── Process images ───────────────────────────────────────
        # Check for jpg/jpeg files first to avoid loading PNG masks as images in Crack500 traincrop
        images = sorted([
            f for f in img_dir.iterdir()
            if f.suffix.lower() in ('.jpg', '.jpeg')
            and ':Zone.Identifier' not in f.name
        ])
        if not images:
            images = sorted([
                f for f in img_dir.iterdir()
                if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
                and ':Zone.Identifier' not in f.name
            ])
        if args.max:
            images = images[:args.max]

        print(f"\n>>> Processing {d['name']} ({len(images)} images) <<<")
        print(f"  Images: {img_dir}")
        print(f"  Masks:  {mask_dir}")
        print(f"  Prefix: {prefix}\n")

        generated = 0
        skipped   = 0
        failed    = 0

        for img_path in images:
            image_id   = f"{prefix}{img_path.stem}"
            logit_file = logits_dir / f"{image_id}_logits.npy"

            # Skip if already done
            if args.resume and logit_file.exists():
                skipped += 1
                continue

            # Load image
            image = cv2.imread(str(img_path))
            if image is None:
                failed += 1
                continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Load binary mask
            mask_path = mask_dir / f"{img_path.stem}.png"
            if not mask_path.exists():
                mask_path = mask_dir / f"{img_path.stem}.bmp"
            if not mask_path.exists():
                failed += 1
                continue

            binary = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            binary = (binary > 127).astype(np.uint8)

            # Split into instances via connected components
            num_labels, labels_map = cv2.connectedComponents(binary)

            all_logits = []
            predictor.set_image(image)

            # Capture encoder features
            features = predictor._features
            image_embed = features["image_embed"].cpu().half().numpy()
            feat0 = features["high_res_feats"][0].cpu().half().numpy()
            feat1 = features["high_res_feats"][1].cpu().half().numpy()

            for label_id in range(1, num_labels):
                instance = (labels_map == label_id).astype(np.uint8)
                if instance.sum() < 50:   # skip tiny noise
                    continue
                box = mask_to_bbox(instance)
                if box is None:
                    continue

                if args.prompt_type == "box_centroid":
                    centroid = mask_to_centroid(instance)
                    point_labels = np.array([1], dtype=np.int32)
                    with torch.no_grad():
                        _, _, logits = predictor.predict(
                            point_coords=centroid,
                            point_labels=point_labels,
                            box=box,
                            multimask_output=False,
                        )
                else:
                    with torch.no_grad():
                        _, _, logits = predictor.predict(
                            box=box,
                            multimask_output=False,
                        )
                all_logits.append(logits[0].astype(np.float32))  # (256, 256)

            if all_logits:
                np.save(str(logit_file), np.stack(all_logits, axis=0))
                
                # Save encoder features in a shared features directory to save disk space
                features_dir = logits_dir.parent / "teacher_features"
                features_dir.mkdir(parents=True, exist_ok=True)
                feat_file = features_dir / f"{image_id}_features.npz"
                if not feat_file.exists():
                    np.savez_compressed(
                        str(feat_file),
                        image_embed=image_embed,
                        feat1=feat1
                    )
                generated += 1
            else:
                failed += 1

            total = generated + skipped + failed
            if total % 50 == 0 or total == len(images):
                print(f"  {total}/{len(images)} | "
                      f"generated={generated} skipped={skipped} failed={failed}")

        print(f"\n  Finished {d['name']}: generated={generated}, skipped={skipped}, failed={failed}")

    print("\n[Done] All active datasets processed successfully.")
    print("Next step:")
    print("  python scripts/run_experiments.py --exp full_kd")


if __name__ == "__main__":
    main()


Writing scripts/generate_teacher_logits.py


In [10]:
%%writefile scripts/combine_datasets.py
#!/usr/bin/env python3
"""
Combine Crack500 and DeepCrack YOLO datasets
============================================
Creates a unified dataset folder `data/datasets/combined_yolo/` by copying files from
`crack500_yolo/` and `deepcrack_yolo/` with dataset prefixes to avoid filename collisions.

Structure of output:
  combined_yolo/
  ├── images/
  │   ├── train/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── train/
  │   ├── val/
  │   └── test/
  └── dataset.yaml
"""

import os
import shutil
import argparse
from pathlib import Path


def combine_split(src_dir: Path, dst_dir: Path, split_name: str, prefix: str, is_label: bool = False):
    """Copy files from src_dir to dst_dir under the given split with a prefix."""
    src_split_dir = src_dir / split_name
    if not src_split_dir.exists():
        print(f"  [Warning] Split directory {src_split_dir} does not exist, skipping.")
        return 0

    dst_split_dir = dst_dir / split_name
    dst_split_dir.mkdir(parents=True, exist_ok=True)

    copied = 0
    exts = ['.txt'] if is_label else ['.jpg', '.jpeg', '.png']

    for file_path in src_split_dir.iterdir():
        if file_path.suffix.lower() in exts and ':Zone.Identifier' not in file_path.name:
            new_name = f"{prefix}{file_path.name}"
            dst_file_path = dst_split_dir / new_name
            if is_label:
                shutil.copy2(file_path, dst_file_path)
            else:
                if dst_file_path.exists() or dst_file_path.is_symlink():
                    try:
                        os.unlink(dst_file_path)
                    except:
                        pass
                os.symlink(file_path, dst_file_path)
            copied += 1

    return copied


def main():
    parser = argparse.ArgumentParser(description="Combine Crack500 and DeepCrack YOLO datasets")
    parser.add_argument(
        "--crack500",
        type=str,
        default="data/datasets/crack500_yolo",
        help="Path to crack500_yolo directory"
    )
    parser.add_argument(
        "--deepcrack",
        type=str,
        default="data/datasets/deepcrack_yolo",
        help="Path to deepcrack_yolo directory"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default="data/datasets/combined_yolo",
        help="Path to combined output directory"
    )
    args = parser.parse_args()

    c500_path = Path(args.crack500).expanduser().resolve()
    dcrack_path = Path(args.deepcrack).expanduser().resolve()
    dst_path = Path(args.dst).expanduser().resolve()

    print(f"[Combine] Crack500 source:  {c500_path}")
    print(f"[Combine] DeepCrack source: {dcrack_path}")
    print(f"[Combine] Destination:      {dst_path}")
    print()

    if not c500_path.exists():
        print(f"ERROR: Crack500 directory not found at {c500_path}")
        return

    if not dcrack_path.exists():
        print(f"ERROR: DeepCrack directory not found at {dcrack_path}")
        return

    if dst_path.exists():
        print(f"[Warning] Output directory exists, clearing: {dst_path}")
        shutil.rmtree(dst_path)

    dst_path.mkdir(parents=True, exist_ok=True)

    splits = ["train", "val", "test"]
    stats = {
        "crack500": {"train": 0, "val": 0, "test": 0},
        "deepcrack": {"train": 0, "val": 0, "test": 0}
    }

    for split in splits:
        print(f"Processing split: {split}")
        
        # 1. Crack500
        # Images
        n_img = combine_split(
            src_dir=c500_path / "images",
            dst_dir=dst_path / "images",
            split_name=split,
            prefix="crack500_",
            is_label=False
        )
        # Labels
        n_lbl = combine_split(
            src_dir=c500_path / "labels",
            dst_dir=dst_path / "labels",
            split_name=split,
            prefix="crack500_",
            is_label=True
        )
        assert n_img == n_lbl, f"Crack500 {split} image/label mismatch: {n_img} vs {n_lbl}"
        stats["crack500"][split] = n_img

        # 2. DeepCrack
        # Images
        n_img = combine_split(
            src_dir=dcrack_path / "images",
            dst_dir=dst_path / "images",
            split_name=split,
            prefix="deepcrack_",
            is_label=False
        )
        # Labels
        n_lbl = combine_split(
            src_dir=dcrack_path / "labels",
            dst_dir=dst_path / "labels",
            split_name=split,
            prefix="deepcrack_",
            is_label=True
        )
        assert n_img == n_lbl, f"DeepCrack {split} image/label mismatch: {n_img} vs {n_lbl}"
        stats["deepcrack"][split] = n_img

    # Write unified dataset.yaml
    yaml_content = f"""# Combined Dataset — Crack500 + DeepCrack
# Auto-generated by combine_datasets.py

path: {dst_path.resolve()}
train: images/train
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats:
# Crack500:  train={stats['crack500']['train']}, val={stats['crack500']['val']}, test={stats['crack500']['test']}
# DeepCrack: train={stats['deepcrack']['train']}, val={stats['deepcrack']['val']}, test={stats['deepcrack']['test']}
# Total:     train={stats['crack500']['train'] + stats['deepcrack']['train']}, val={stats['crack500']['val'] + stats['deepcrack']['val']}, test={stats['crack500']['test'] + stats['deepcrack']['test']}
"""
    with open(dst_path / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\ndataset.yaml written to {dst_path / 'dataset.yaml'}")

    # Verify counts
    print("\n[Verify] Checking combined dataset...")
    for split in splits:
        img_dir = dst_path / "images" / split
        lbl_dir = dst_path / "labels" / split
        imgs = list(img_dir.glob("*"))
        lbls = list(lbl_dir.glob("*"))
        print(f"  {split}: {len(imgs)} images, {len(lbls)} labels")
        assert len(imgs) == len(lbls), f"Mismatch in combined {split} split!"

    print("\n✓ Combined dataset successfully created!")


if __name__ == "__main__":
    main()


Writing scripts/combine_datasets.py


In [11]:
%%writefile scripts/test_model.py
#!/usr/bin/env python3
"""
Test the KD-trained student model.
Shows predictions on test images with metrics.

Usage:
  # Test on crack500 test set
  python scripts/test_model.py

  # Test on a single image
  python scripts/test_model.py --image path/to/image.jpg

  # Test on a folder
  python scripts/test_model.py --folder path/to/images/
"""

import argparse
import random
import time
import cv2
import numpy as np
from pathlib import Path

MODEL_PATH  = Path.home() / "distill/runs/kd_demo/kd_student/weights/best.pt"
TEST_DIR    = Path.home() / "distill/data/datasets/crack500/testcrop"
OUTPUT_DIR  = Path.home() / "distill/runs/kd_demo/test_results"


def test(source, model_path=MODEL_PATH, max_images=20):
    from ultralytics import YOLO

    model = YOLO(str(model_path))
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Collect images
    if isinstance(source, str) and Path(source).is_file():
        images = [Path(source)]
    elif isinstance(source, str) and Path(source).is_dir():
        all_images = [
            f for f in Path(source).iterdir()
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
            and ':Zone' not in f.name
            and not f.name.lower().endswith('_mask.png')
        ]
        images = random.sample(all_images, min(max_images, len(all_images)))
    else:
        images = source[:max_images]

    print(f"\nTesting on {len(images)} images...")
    print(f"Model: {model_path}\n")

    times = []
    detections = []

    for img_path in images:
        t0 = time.perf_counter()
        results = model.predict(
            str(img_path),
            imgsz=512,
            conf=0.25,
            device="cuda",
            verbose=False,
        )
        ms = (time.perf_counter() - t0) * 1000
        times.append(ms)

        r = results[0]
        n_instances = len(r.boxes) if r.boxes is not None else 0
        detections.append(n_instances)

        # Save annotated image
        annotated = r.plot()
        out_path  = OUTPUT_DIR / img_path.name
        cv2.imwrite(str(out_path), annotated)

        print(f"  {img_path.name:40s}  {n_instances} cracks  {ms:.1f}ms")

    # Summary
    avg_ms  = np.mean(times)
    avg_fps = 1000 / avg_ms
    print(f"\n{'─'*55}")
    print(f"  Images tested:    {len(images)}")
    print(f"  Avg speed:        {avg_ms:.1f} ms  →  {avg_fps:.0f} FPS")
    print(f"  Avg detections:   {np.mean(detections):.1f} instances/image")
    print(f"  Results saved to: {OUTPUT_DIR}")
    print(f"{'─'*55}\n")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--image",  type=str, default=None)
    parser.add_argument("--folder", type=str, default=None)
    parser.add_argument("--model",  type=str, default=str(MODEL_PATH))
    parser.add_argument("--n",      type=int, default=20, help="Max images to test")
    parser.add_argument("--val",    action="store_true", help="Run full validation (eval metrics)")
    parser.add_argument("--data",   type=str, default="data/datasets/crack500_uncropped_yolo/dataset.yaml", help="Path to dataset.yaml")
    args = parser.parse_args()

    if args.val:
        from ultralytics import YOLO
        print(f"\nRunning validation evaluation on: {args.data}")
        print(f"Model: {args.model}\n")
        model = YOLO(args.model)
        metrics = model.val(
            data=args.data,
            imgsz=512,
            device="cuda",
            verbose=True,
        )
        print("\n[Validation Metrics]")
        print(f"  mAP50-box:    {metrics.box.map50:.4f}")
        print(f"  mAP50-seg:    {metrics.seg.map50:.4f}")
        print(f"  mAP50-95-seg: {metrics.seg.map:.4f}")
    else:
        source = args.image or args.folder or str(TEST_DIR)
        test(source, model_path=args.model, max_images=args.n)


if __name__ == "__main__":
    main()


Writing scripts/test_model.py


In [12]:
%%writefile scripts/run_experiments.py
#!/usr/bin/env python3
"""
Run all Crack-Distill experiments in sequence.
Paper experiments in the correct order:
  1. baseline_finetune  — lower bound
  2. pseudo_labels      — SAM quality alone
  3. full_kd            — main result
  4-7. low_data_*       — H2 hypothesis
  8. robustness         — deployment claim

Usage:
  python scripts/run_experiments.py --exp all
  python scripts/run_experiments.py --exp full_kd
  python scripts/run_experiments.py --exp baseline_finetune
"""

import argparse
import sys
import os

project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, project_root)
os.environ["PYTHONPATH"] = project_root + os.pathsep + os.environ.get("PYTHONPATH", "")

from utils.config_loader import load_config, override_config
from distillation.trainer import CrackDistillTrainer


EXPERIMENTS = {
    "baseline_finetune": {
        "description": "Lower bound — YOLO11 fine-tuned, no KD",
        "overrides": {
            "distillation.enabled": False,
        }
    },
    "pseudo_labels": {
        "description": "SAM quality alone — hard pseudo-labels, no soft KD",
        "overrides": {
            "distillation.enabled": True,
            "distillation.losses.mask_kd.enabled": False,
            "distillation.losses.feature.enabled": False,
            "distillation.losses.boundary.enabled": False,
        }
    },
    "full_kd_box": {
        "description": "Full KD pipeline — Bounding box only prompts",
        "overrides": {
            "distillation.enabled": True,
            "distillation.losses.mask_kd.enabled": True,
            "distillation.losses.feature.enabled": True,
            "distillation.losses.boundary.enabled": True,
            "teacher.logits_dir": "data/teacher_logits_box/",
        }
    },
    "full_kd_centroid": {
        "description": "Full KD pipeline — Bounding box + Centroid point prompts",
        "overrides": {
            "distillation.enabled": True,
            "distillation.losses.mask_kd.enabled": True,
            "distillation.losses.feature.enabled": True,
            "distillation.losses.boundary.enabled": True,
            "teacher.logits_dir": "data/teacher_logits_centroid/",
        }
    },
    "low_data_5pct": {
        "description": "H2: KD in low-data regime (5% of training set)",
        "overrides": {
            "distillation.enabled": True,
            "data.train_fraction": 0.05,
        }
    },
    "low_data_10pct": {
        "description": "H2: KD in low-data regime (10%)",
        "overrides": {
            "distillation.enabled": True,
            "data.train_fraction": 0.10,
        }
    },
    "low_data_25pct": {
        "description": "H2: KD in low-data regime (25%)",
        "overrides": {
            "distillation.enabled": True,
            "data.train_fraction": 0.25,
        }
    },
    "low_data_50pct": {
        "description": "H2: KD in low-data regime (50%)",
        "overrides": {
            "distillation.enabled": True,
            "data.train_fraction": 0.50,
        }
    },
    # Ablation: remove each KD component one at a time
    "ablation_no_mask_kd": {
        "description": "Ablation: full KD minus mask KD loss",
        "overrides": {
            "distillation.enabled": True,
            "distillation.losses.mask_kd.enabled": False,
            "distillation.losses.feature.enabled": True,
            "distillation.losses.boundary.enabled": True,
        }
    },
    "ablation_no_feature": {
        "description": "Ablation: full KD minus feature distillation",
        "overrides": {
            "distillation.enabled": True,
            "distillation.losses.mask_kd.enabled": True,
            "distillation.losses.feature.enabled": False,
            "distillation.losses.boundary.enabled": True,
        }
    },
    "ablation_no_boundary": {
        "description": "Ablation: full KD minus boundary loss",
        "overrides": {
            "distillation.enabled": True,
            "distillation.losses.mask_kd.enabled": True,
            "distillation.losses.feature.enabled": True,
            "distillation.losses.boundary.enabled": False,
        }
    },
}


def run_experiment(exp_name: str, cfg_path: str = "configs/config.yaml"):
    exp = EXPERIMENTS[exp_name]
    print(f"\n{'='*60}")
    print(f"Experiment: {exp_name}")
    print(f"Description: {exp['description']}")
    print(f"Overrides: {exp['overrides']}")
    print(f"{'='*60}\n")

    # Load base config and apply overrides
    cfg = load_config(cfg_path)
    cfg = override_config(cfg, exp["overrides"])
    cfg = override_config(cfg, {"project.name": "crack_distill", "project.experiment": exp_name})

    # Run training — pass overridden config directly
    trainer = CrackDistillTrainer(cfg_path, override_cfg=cfg)
    trainer.train()
    results = trainer.test()

    return {exp_name: results}


def main():
    parser = argparse.ArgumentParser(description="Run Crack-Distill experiments")
    parser.add_argument(
        "--exp",
        type=str,
        default="full_kd_centroid",
        choices=list(EXPERIMENTS.keys()) + ["all", "paper_main", "ablation"],
        help="Experiment to run"
    )
    parser.add_argument("--cfg", type=str, default="configs/config.yaml")
    args = parser.parse_args()

    if args.exp == "all":
        exps = list(EXPERIMENTS.keys())
    elif args.exp == "paper_main":
        exps = ["baseline_finetune", "pseudo_labels", "full_kd_box", "full_kd_centroid"]
    elif args.exp == "ablation":
        exps = ["ablation_no_mask_kd", "ablation_no_feature", "ablation_no_boundary"]
    else:
        exps = [args.exp]

    all_results = {}
    for exp_name in exps:
        results = run_experiment(exp_name, args.cfg)
        all_results.update(results)
        print(f"\n✓ {exp_name} done: {results}\n")

    print("\n" + "="*60)
    print("ALL RESULTS SUMMARY")
    print("="*60)
    for exp_name, metrics in all_results.items():
        print(f"\n{exp_name}:")
        for k, v in metrics.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")


if __name__ == "__main__":
    main()


Writing scripts/run_experiments.py


In [13]:
%%writefile scripts/tune_kd_weights.py
#!/usr/bin/env python3
"""
Optuna KD Weight Tuning Script — Crack-Distill
=============================================
Tunes KD loss weights (mask_kd, feature, boundary) and temperature
to maximize student validation performance (mAP50-seg).
Cleans up trial run directories to prevent disk exhaustion.
"""

import argparse
import sys
import os
import shutil
import json
from pathlib import Path
import torch

# Add project root to path and environment for subprocesses
project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, project_root)
os.environ["PYTHONPATH"] = project_root + os.pathsep + os.environ.get("PYTHONPATH", "")

from utils.config_loader import load_config, override_config
from distillation.trainer import CrackDistillTrainer

try:
    import optuna
except ImportError:
    print("Optuna is not installed. Please run: pip install optuna")
    sys.exit(1)


def objective(trial, args, base_cfg) -> float:
    # 1. Suggest hyperparameters
    temp = trial.suggest_float("temperature", 2.0, 4.0)
    w_mask = trial.suggest_float("mask_kd", 0.5, 2.5)
    w_feat = trial.suggest_float("feature", 0.5, 2.0)
    w_bound = trial.suggest_float("boundary", 0.5, 3.0)

    print(f"\n--- Starting Trial {trial.number} ---")
    print(f"Suggested parameters:")
    print(f"  temperature: {temp:.4f}")
    print(f"  mask_kd weight: {w_mask:.4f}")
    print(f"  feature weight: {w_feat:.4f}")
    print(f"  boundary weight: {w_bound:.4f}")

    # 2. Setup overrides
    overrides = {
        "distillation.enabled": True,
        "distillation.temperature": temp,
        "distillation.losses.mask_kd.enabled": True,
        "distillation.losses.mask_kd.weight": w_mask,
        "distillation.losses.feature.enabled": True,
        "distillation.losses.feature.weight": w_feat,
        "distillation.losses.boundary.enabled": True,
        "distillation.losses.boundary.weight": w_bound,
        "train.epochs": args.epochs,
        "teacher.logits_dir": "data/teacher_logits_centroid/",
        "data.train_fraction": args.train_fraction,
    }
    
    # Run name unique per trial
    exp_name = f"optuna_trial_{trial.number}"
    overrides.update({
        "project.name": "crack_distill",
        "project.experiment": exp_name
    })

    # Apply overrides
    cfg = override_config(base_cfg, overrides)

    # Instantiate trainer and attach trial
    trainer = CrackDistillTrainer(args.cfg, override_cfg=cfg)
    trainer.optuna_trial = trial
    
    score = 0.0
    try:
        # Run training
        trainer.train()
        
        # Evaluate model on validation
        results = trainer.test()
        score = results.get("mAP50-seg", 0.0)
        
        # Check if this trial is the best so far
        is_best = False
        try:
            best_trial = trial.study.best_trial
            if score > best_trial.value:
                is_best = True
        except ValueError:
            # First trial completed
            is_best = True
            
        if is_best:
            # Save best parameters to json
            best_params_path = Path("runs/best_optuna_params.json")
            best_params_path.parent.mkdir(parents=True, exist_ok=True)
            best_info = {
                "trial_number": trial.number,
                "score": score,
                "parameters": {
                    "temperature": temp,
                    "mask_kd": w_mask,
                    "feature": w_feat,
                    "boundary": w_bound
                }
            }
            with open(best_params_path, "w") as f:
                json.dump(best_info, f, indent=2)
                
            # Copy best model weight
            best_model_src = trainer.best_pt
            if best_model_src.exists():
                best_model_dst = Path("runs/optuna_best_model.pt")
                shutil.copy2(best_model_src, best_model_dst)
                print(f"[Optuna] New best trial {trial.number}! Score (mAP50-seg): {score:.4f}. Saved best weights to {best_model_dst}")
                
    except optuna.exceptions.TrialPruned:
        print(f"[Optuna] Trial {trial.number} was pruned early.")
        raise
    except Exception as e:
        print(f"[Optuna] Trial {trial.number} failed with exception: {e}")
        import traceback
        traceback.print_exc()
        score = 0.0
    finally:
        # 3. Clean up runs to save disk space
        print(f"[Optuna] Cleaning up Trial {trial.number} run directories...")
        run_dirs_to_clean = [
            trainer.run_dir,
            trainer.run_dir.parent / f"{trainer.run_dir.name}_stage1",
            trainer.run_dir.parent / f"{trainer.run_dir.name}_stage2"
        ]
        for d in run_dirs_to_clean:
            # Delete direct directories
            if d.exists():
                try:
                    shutil.rmtree(d)
                except Exception as err:
                    print(f"[Optuna] Warning: Failed to delete {d}: {err}")
            
            # Clean any wildcard matches (YOLO sometimes appends suffixes like 2, 3...)
            for p in trainer.run_dir.parent.glob(f"{d.name}*"):
                if p.exists() and p.is_dir():
                    try:
                        shutil.rmtree(p)
                    except Exception:
                        pass
        
        # Clear CUDA memory cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    return score


def main():
    parser = argparse.ArgumentParser(description="Tune KD weights using Optuna")
    parser.add_argument("--cfg", type=str, default="configs/config.yaml")
    parser.add_argument("--trials", type=int, default=10, help="Number of Optuna trials")
    parser.add_argument("--epochs", type=int, default=5, help="Number of training epochs per trial")
    parser.add_argument("--train-fraction", type=float, default=0.20, help="Fraction of training data to use for tuning")
    parser.add_argument("--study-name", type=str, default="kd_weight_tuning")
    parser.add_argument("--storage", type=str, default=None, help="Database URL for Optuna storage (optional)")
    args = parser.parse_args()

    # Load base config
    base_cfg = load_config(args.cfg)

    # Set up Optuna logging verbosity
    optuna.logging.set_verbosity(optuna.logging.INFO)

    study = optuna.create_study(
        study_name=args.study_name,
        direction="maximize",
        storage=args.storage,
        load_if_exists=True,
        pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=5)
    )

    print(f"Starting study '{args.study_name}' with {args.trials} trials, each training for {args.epochs} epochs.")
    
    study.optimize(lambda trial: objective(trial, args, base_cfg), n_trials=args.trials)

    print("\n" + "="*60)
    print("OPTUNA TUNING COMPLETED")
    print("="*60)
    try:
        print(f"Best Trial: #{study.best_trial.number}")
        print(f"Best Score (mAP50-seg): {study.best_value:.4f}")
        print("Best Parameters:")
        for k, v in study.best_params.items():
            print(f"  {k}: {v:.4f}")
    except ValueError:
        print("No trials completed successfully.")
    print("="*60)


if __name__ == "__main__":
    main()


Writing scripts/tune_kd_weights.py


## 🛠️ Environment Setup & SAM 2 Installation

In [14]:
# Install required packages
!pip install -q ultralytics albumentations pycocotools thop pyyaml optuna

# Clone and install SAM 2 package (cloned as sam2_repo to prevent package shadowing errors)
!git clone https://github.com/facebookresearch/sam2.git sam2_repo || true
%cd sam2_repo
!pip install -e .
%cd ..

# Clean up any old shadowing 'sam2' directories if they exist from previous runs
!rm -rf sam2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 82.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23

## 📁 Writable Dataset Linker & Diagnostics

In [15]:
import os
import shutil
from pathlib import Path

input_dir = Path("/kaggle/input")
datasets_dir = Path("data/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)
checkpoints_dir = Path("checkpoints")
checkpoints_dir.mkdir(parents=True, exist_ok=True)

# 1. Clean up any existing local dataset directories/symlinks to avoid read-only collisions
for folder in ["combined_yolo", "crack500_yolo", "crack500_uncropped_yolo", "deepcrack_yolo", "crack500", "deepcrack"]:
    p = datasets_dir / folder
    if os.path.lexists(p):
        if os.path.islink(p): os.unlink(p)
        else: shutil.rmtree(p)

# 2. Clean up teacher logits folders and link them to /tmp to redirect disk usage
for folder in ["teacher_logits_box", "teacher_logits_centroid", "teacher_features"]:
    p_local = Path("data") / folder
    p_tmp = Path("/tmp") / folder
    if os.path.lexists(p_local):
        if os.path.islink(p_local): os.unlink(p_local)
        else: shutil.rmtree(p_local)
    p_tmp.mkdir(parents=True, exist_ok=True)
    os.symlink(p_tmp, p_local)
    print(f"Redirected {p_local} -> {p_tmp}")

# 3. Locate and link SAM 2 weights
linked_sam = False
for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    if "sam2_hiera_large.pt" in files and not linked_sam:
        dest = checkpoints_dir / "sam2_hiera_large.pt"
        if os.path.lexists(dest): os.remove(dest)
        os.symlink(root_path / "sam2_hiera_large.pt", dest)
        print(f"Linked SAM 2 checkpoint: {root_path / 'sam2_hiera_large.pt'} -> {dest}")
        linked_sam = True

if not (checkpoints_dir / "sam2_hiera_large.pt").exists():
    print("SAM 2 checkpoint not found in inputs. Downloading...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt -O checkpoints/sam2_hiera_large.pt

# 4. Link raw datasets for conversion
linked_crack = False
linked_deep = False
for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    if "traincrop" in dirs and not linked_crack:
        dest = datasets_dir / "crack500"
        os.symlink(root_path, dest)
        print(f"Linked raw Crack500: {root_path} -> {dest}")
        linked_crack = True
    if "train_img" in dirs and not linked_deep:
        dest = datasets_dir / "deepcrack"
        os.symlink(root_path, dest)
        print(f"Linked raw DeepCrack: {root_path} -> {dest}")
        linked_deep = True

if not (datasets_dir / "crack500").exists():
    print("WARNING: Raw Crack500 dataset not linked!")
if not (datasets_dir / "deepcrack").exists():
    print("WARNING: Raw DeepCrack dataset not linked!")

Redirected data/teacher_logits_box -> /tmp/teacher_logits_box
Redirected data/teacher_logits_centroid -> /tmp/teacher_logits_centroid
Redirected data/teacher_features -> /tmp/teacher_features
SAM 2 checkpoint not found in inputs. Downloading...
Linked raw DeepCrack: /kaggle/input/datasets/rauffatali/distill-datasetforme/deepcrack -> data/datasets/deepcrack
Linked raw Crack500: /kaggle/input/datasets/rauffatali/distill-datasetforme/crack500 -> data/datasets/crack500


## ⚡ Data Conversion & Teacher Logits Generation

Convert datasets to YOLO format, generate teacher logits, and combine them. By default, we use **box_centroid** logits for tuning.

In [16]:
# Convert raw datasets to YOLO format
!python scripts/convert_crack500.py --src data/datasets/crack500 --dst data/datasets/crack500_yolo
!python scripts/convert_crack500_uncropped.py --src data/datasets/crack500 --dst data/datasets/crack500_uncropped_yolo
!python scripts/convert_deepcrack.py --src data/datasets/deepcrack --dst data/datasets/deepcrack_yolo

# Generate teacher logits (tuning is performed using box+centroid guides)
print("=== Generating Box+Centroid Logits for Tuning ===")
!python scripts/generate_teacher_logits.py --prompt-type box_centroid --logits-dir data/teacher_logits_centroid

# Combine datasets
!python scripts/combine_datasets.py

[Convert] Source: /kaggle/input/datasets/rauffatali/distill-datasetforme/crack500
[Convert] Output: /kaggle/working/data/datasets/crack500_yolo

  train: 1896 images converted, 0 skipped
  val: 348 images converted, 0 skipped
  test: 1124 images converted, 0 skipped

  dataset.yaml written to /kaggle/working/data/datasets/crack500_yolo/dataset.yaml

[Verify] Checking converted dataset...
  train: 1896 images ✓
    labels with cracks: 1896 | empty (no crack): 0
  val: 348 images ✓
    labels with cracks: 348 | empty (no crack): 0
  test: 1124 images ✓
    labels with cracks: 1124 | empty (no crack): 0

  ✓ Dataset looks good!

[Done] Converted dataset at: /kaggle/working/data/datasets/crack500_yolo

Next step — test YOLO11 loads it:
  from ultralytics import YOLO
  model = YOLO('yolo11n-seg.pt')
  model.train(data='/kaggle/working/data/datasets/crack500_yolo/dataset.yaml', epochs=1, imgsz=512)
[Convert] Source: /kaggle/input/datasets/rauffatali/distill-datasetforme/crack500
[Convert] Ou

## 🔍 Hyperparameter Tuning with Optuna

Runs the Optuna weight search. Tries different values of distillation loss weights and temperature. Results and checkpoints are stored dynamically.

In [17]:
# Install optuna
!pip install -q optuna

# Run the tuning script (10 trials of 5 epochs each)
!python scripts/tune_kd_weights.py --trials 10 --epochs 15 --train-fraction 0.20


[I 2026-07-11 08:01:55,859] A new study created in memory with name: kd_weight_tuning
Starting study 'kd_weight_tuning' with 10 trials, each training for 15 epochs.

--- Starting Trial 0 ---
Suggested parameters:
  temperature: 3.0220
  mask_kd weight: 2.2698
  feature weight: 1.5419
  boundary weight: 2.4235
[Trainer] Task:       instance_seg
[Trainer] Student:    yolov8n-seg
[Trainer] KD enabled: True
[Trainer] Epochs:     15
[Trainer] Data:       /kaggle/working/data/datasets/combined_yolo/dataset.yaml
[Trainer] Run dir:    /kaggle/working/runs/crack_distill_optuna_trial_0_instance_seg_yolov8n-seg
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[Trainer] Progressive Distillation Enabled!
[Trainer] Total Epochs: 15
[Trainer] 

## 📊 Evaluate Best Optuna Model

In [18]:
print("=== Best Tuned KD Performance (Cropped) ===")
!python scripts/test_model.py --val --model runs/optuna_best_model.pt --data data/datasets/combined_yolo/dataset.yaml

print("=== Best Tuned KD Performance (Uncropped) ===")
!python scripts/test_model.py --val --model runs/optuna_best_model.pt --data data/datasets/crack500_uncropped_yolo/dataset.yaml

=== Best Tuned KD Performance (Cropped) ===

Running validation evaluation on: data/datasets/combined_yolo/dataset.yaml
Model: runs/optuna_best_model.pt

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8n-seg summary (fused): 89 layers, 3,359,187 parameters, 0 gradients, 12.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1628.2±446.7 MB/s, size: 54.2 KB)
val: Scanning /kaggle/working/data/datasets/combined_yolo/labels/val.cache... 408 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 408/408 81.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 6.2it/s 4.2s
                   all        408        799       0.68      0.462      0.523      0.339      0.726      0.423      0.495      0.181
Speed: 0.6ms preprocess, 2.6ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /kaggle/working/runs/segment/val